```{contents}
```

## Vanishing Gradient Problem

### Definition

The **vanishing gradient problem** occurs when gradients propagated backward through a deep neural network become **exponentially small**, preventing early layers from learning effectively. As a result, the network converges very slowly or fails to learn meaningful representations.

---

### Intuition

During backpropagation, gradients are repeatedly multiplied by weights and activation derivatives:

$$
\frac{\partial L}{\partial W_1}
= \frac{\partial L}{\partial a_L}
\prod_{k=2}^{L} \frac{\partial a_k}{\partial a_{k-1}}
$$

If these derivatives are mostly **less than 1**, their product shrinks exponentially as depth increases.

**Consequences:**

* Shallow layers receive almost no learning signal.
* Deep networks behave like shallow ones.
* Training stagnates.

---

### Where It Appears Most

| Architecture               | Why                                |
| -------------------------- | ---------------------------------- |
| Sigmoid / Tanh networks    | Saturating derivatives near 0      |
| Very deep feedforward nets | Long gradient chains               |
| RNNs                       | Temporal depth amplifies the issue |

---

### Visualization of Gradient Flow

```
Output → Backprop → Layer N → Layer N-1 → ... → Layer 1
          0.8      0.6        0.4        0.0003
```

The earliest layers effectively stop learning.

---

### Mathematical Cause

For sigmoid activation:

$$
\sigma'(x) = \sigma(x)(1 - \sigma(x)) \le 0.25
$$

After 20 layers:

$$
0.25^{20} \approx 9.1 \times 10^{-13}
$$

Practically zero.

---

### Training Workflow Impact

1. Forward pass computes activations.
2. Loss is computed.
3. Backward pass propagates gradients.
4. Early layers receive near-zero gradients.
5. Weight updates become negligible.
6. Model underfits.

---

### Remediation Strategies

| Technique                  | How It Helps                       |
| -------------------------- | ---------------------------------- |
| ReLU / LeakyReLU           | Avoids saturation                  |
| He / Xavier initialization | Preserves gradient variance        |
| Batch Normalization        | Stabilizes activation distribution |
| Residual connections       | Create direct gradient paths       |
| LSTM / GRU (for RNNs)      | Gated gradient flow                |
| Gradient clipping          | Controls exploding / stabilizing   |

---

### PyTorch Demonstration

#### Failure Case: Sigmoid Deep Network

```python
import torch
import torch.nn as nn

model = nn.Sequential(*[nn.Linear(128, 128), nn.Sigmoid()] * 20)

x = torch.randn(64, 128)
y = torch.randn(64, 128)

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

loss = loss_fn(model(x), y)
loss.backward()

for i, layer in enumerate(model):
    if isinstance(layer, nn.Linear):
        print(f"Layer {i} grad norm:", layer.weight.grad.norm().item())
```

**Observed behavior:** early layers show gradient norms approaching zero.

---

#### Fixed Case: ReLU + He Initialization

```python
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)

model = nn.Sequential(*[nn.Linear(128, 128), nn.ReLU()] * 20)
model.apply(init_weights)
```

Gradients now propagate stably.

---

### Residual Connection Illustration

```python
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return x + self.net(x)
```

Residual paths allow gradients to bypass nonlinearities.

---

### Practical Diagnosis Checklist

| Symptom                       | Likely Cause            |
| ----------------------------- | ----------------------- |
| Loss stops decreasing         | Vanishing gradients     |
| Early layers’ gradients ~0    | Saturating activations  |
| Deeper networks perform worse | Depth-induced vanishing |

---

### Summary

| Aspect          | Explanation                           |
| --------------- | ------------------------------------- |
| Root cause      | Exponential decay of gradients        |
| Main risk       | Early layers stop learning            |
| Most vulnerable | Sigmoid, tanh, deep RNNs              |
| Core fixes      | ReLU, normalization, skip connections |
| Modern solution | Residual & gated architectures        |

---

This problem motivated the development of modern architectures such as **ResNet**, **LSTM**, and **Transformers**, where gradient flow is explicitly engineered.
